# Titanic: построение и оценка моделей 
Цель: подготовить признаки на основе выводов из EDA, обучить базовые модели и зафиксировать стартовую метрику (`Accuracy`) на кросс-валидации.

In [1]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
    rare_titles = df["Title"].value_counts()[df["Title"].value_counts() < 10].index
    df["Title"] = df["Title"].replace(rare_titles, "Rare")
    df["Title"] = df["Title"].replace({"Mile" : "Miss", "Ms": "Miss", "Mme" : "Mrs"})

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)


    df["HasCabin"] = df["Cabin"].notnull().astype(int)  

    return df

train_feat = add_features(train)
test_feat = add_features(test)

## Обработка пропусков и создание взаимодействий
- Пропуски в `Age` заполняем медианным возрастом соответствующего обращения (`Title`), считая медианы строго по `train` во избежание утечки данных.
- Добавляем бинарный признак `IsChild` (возраст `<= 6` лет) и комбинированный признак `Sex_Pclass` для учета разного влияния класса каюты на мужчин и женщин.

In [7]:
title_age_medians = train_feat.groupby("Title")["Age"].median()
for title, median_age in title_age_medians.items():
    train_feat.loc[(train_feat["Age"].isnull()) & (train_feat["Title"] == title), "Age"] = median_age
    test_feat.loc[(test_feat["Age"].isnull()) & (test_feat["Title"] == title), "Age"] = median_age
    
global_age_median = train_feat["Age"].median()
train_feat["Age"] = train_feat["Age"].fillna(global_age_median)
test_feat["Age"] = test_feat["Age"].fillna(global_age_median)


train_feat["IsChild"] = (train_feat["Age"] <= 6).astype(int)
test_feat["IsChild"] = (test_feat["Age"] <= 6).astype(int)

train_feat["Sex_Pclass"] = train_feat["Sex"] + "_" + train_feat["Pclass"].astype(str)
test_feat["Sex_Pclass"] = test_feat["Sex"] + "_" + test_feat["Pclass"].astype(str)

num_features = ["Age", "Fare", "FamilySize", "IsAlone", "HasCabin", "IsChild"]
cat_features = ["Pclass", "Sex", "Embarked", "Title", "Sex_Pclass"]

X = train_feat[num_features + cat_features]
y = train_feat["Survived"]
X_test = test_feat[num_features + cat_features]
